# AD Knowledge Graph — Exploration Notebook

This notebook connects to the Neo4j knowledge graph and runs exploratory queries
to identify drug repurposing candidates for Alzheimer's disease.

**Pre-requisites:**
1. Docker: `docker compose up -d`
2. Pipeline ran: `python -m ad_kg ingest && python -m ad_kg extract && python -m ad_kg resolve && python -m ad_kg load`
3. Install extras: `pip install networkx matplotlib jupyter`

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

# Load .env from project root
load_dotenv(Path('..') / '.env')

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

print(f'Neo4j URI: {NEO4J_URI}')

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j successfully.')

In [ ]:
# Quick node count overview
with driver.session() as session:
    result = session.run("""
        MATCH (n)
        RETURN labels(n)[0] AS label, count(n) AS count
        ORDER BY count DESC
    """)
    for row in result:
        print(f"{row['label']}: {row['count']} nodes")

## Query 1: Whitespace Opportunity

Drugs with FAERS protective signal (ROR < 1.0) and bridge gene support but **no active AD trial**.
These represent the most actionable repurposing candidates.

In [ ]:
import sys
sys.path.insert(0, str(Path('..') / 'src'))

from ad_kg.graph.queries import QUERIES, run_query

results = run_query(driver, 'whitespace_opportunity')

print(f'Found {len(results)} whitespace opportunity candidates:')
for row in results[:20]:
    print(f"  - {row.get('drug')} ({row.get('drug_id', '')})") 

In [ ]:
# All 11 queries in one place
print('Available named queries:')
for name in QUERIES:
    print(f'  - {name}')

## Query 2: Protective Drugs Ranked

In [ ]:
protective = run_query(driver, 'protective_drugs_ranked')
print(f'Found {len(protective)} protective drugs:')
for row in protective[:15]:
    print(f"  {row.get('drug'):30s}  ROR={row.get('best_ror', 'N/A'):.3f}  lit={row.get('literature_mentions', 0)}")

## Query 3: Bridge Genes (GWAS overlap AD + metabolic)

In [ ]:
bridge_genes = run_query(driver, 'bridge_genes_ranked')
print(f'Found {len(bridge_genes)} bridge genes:')
for row in bridge_genes[:15]:
    print(f"  {row.get('gene'):15s}  AD p={row.get('min_ad_pval', 'N/A')}  Met p={row.get('min_metabolic_pval', 'N/A')}")

## Network Visualization: Semaglutide neighborhood

In [ ]:
try:
    import networkx as nx
    import matplotlib.pyplot as plt

    neighbors = run_query(driver, 'semaglutide_neighbors')

    G = nx.DiGraph()
    G.add_node('semaglutide', node_type='Drug')

    for row in neighbors:
        name = row.get('name') or str(row.get('node_type', 'unknown'))
        ntype = row.get('node_type', 'Entity')
        G.add_node(name, node_type=ntype)
        G.add_edge('semaglutide', name)

    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=0.5, seed=42)

    # Color nodes by type
    color_map = {
        'Drug': '#4CAF50',
        'Gene': '#2196F3',
        'Disease': '#F44336',
        'Paper': '#FF9800',
        'Trial': '#9C27B0',
        'SNP': '#00BCD4',
    }
    node_colors = [
        color_map.get(G.nodes[n].get('node_type', 'Entity'), '#9E9E9E')
        for n in G.nodes()
    ]

    nx.draw_networkx(
        G, pos,
        node_color=node_colors,
        node_size=300,
        font_size=7,
        arrows=True,
        edge_color='#cccccc',
        alpha=0.9,
    )
    plt.title('Semaglutide Knowledge Graph Neighborhood (2 hops)', fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('../data/semaglutide_network.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
except ImportError:
    print('Install networkx and matplotlib: pip install networkx matplotlib')

In [ ]:
driver.close()
print('Connection closed.')